## Car Price Prediction

## 1) Problem statement.
This dataset comprises used cars sold on cardehko.com in India as well as important features
of these cars.
If user can predict the price of the car based on input features.
Prediction results can be used to give new seller the price suggestion based on market
condition.

## 2) Data Collection.
The Dataset is collected from scrapping from cardheko webiste
The data consists of 13 column and 15411 rows.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
df = pd.read_csv('cardekho_imputated.csv',index_col=[0])

In [3]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning

In [4]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)

In [6]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [7]:
# ALl different types of features
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical features:', len(num_features))

cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorial features:', len(cat_features))

discrete_feature = [feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Descrete features:',len(discrete_feature))

continous_feature = [feature for feature in num_features if feature not in discrete_feature]
print('Num of continous feature:',len(continous_feature))


Num of Numerical features: 7
Num of Categorial features: 4
Num of Descrete features: 2
Num of continous feature: 5


## Train Test Split


In [8]:
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'],axis=1)
y = df['selling_price']

In [9]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


## Feature Encoding and Scaling

In [10]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [11]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [12]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [13]:
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

In [14]:
# Create column transformer with 3 types of transformers 
num_features = X.select_dtypes(exclude='object').columns
onehot_columns = ['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_transformer = StandardScaler()
Onehot_transformer = OneHotEncoder()

preprocessor = ColumnTransformer([
    ('OneHotEncoder',Onehot_transformer,onehot_columns),
    ('StandardScaler',num_transformer,num_features)
],remainder='passthrough')



In [15]:
X = preprocessor.fit_transform(X)

In [16]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.30,random_state=42)

In [18]:
X_train

array([[ 1.        ,  0.        ,  0.        , ...,  0.02291783,
        -0.27665384, -0.40302241],
       [ 0.        ,  1.        ,  0.        , ..., -0.17282578,
        -0.2759557 , -0.40302241],
       [ 0.        ,  1.        ,  0.        , ..., -0.0480872 ,
        -0.39068268, -0.40302241],
       ...,
       [ 0.        ,  1.        ,  0.        , ..., -0.9366097 ,
        -0.78070786, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43582879, -0.40302241],
       [ 0.        ,  1.        ,  0.        , ..., -0.04616815,
         0.06194201, -0.40302241]], shape=(10787, 17))

## Model Training And Selection

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,root_mean_squared_error

In [20]:
# Create a function to evaluate Model
def evaluate_model(true,predicted):
    r2 = r2_score(true,predicted)
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = root_mean_squared_error(true,predicted)
    return r2,mae,mse,rmse

In [40]:
## Begining of training of Model
models = {
    'RandomForestRegressor':RandomForestRegressor(),
    'Linear Regression':LinearRegression(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'KNeighborsRegressor':KNeighborsRegressor(),
    'DecisionTreeRegressor':DecisionTreeRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)  # Train Model

    #make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #evaluate train test dataset
    model_train_r2, model_train_mae, model_train_mse, model_train_rmse= evaluate_model(y_train,y_train_pred)
    model_test_r2, model_test_mae, model_test_mse,model_test_rmse = evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])

    print('model performance for training set')
    print(f"Root Mean Squared error {model_train_rmse:.4f}")
    print(f"Mean Squared error {model_train_mse:.4f}")
    print(f"Mean Absolute error {model_train_mae:.4f}")
    print(f"R2 Square {model_train_r2:.4f}")

    print('='*35)

    print('model performance for testing set')
    print(f"Root Mean Squared error {model_test_rmse:.4f}")
    print(f"Mean Squared error {model_test_mse:.4f}")
    print(f"Mean Absolute error {model_test_mae:.4f}")
    print(f"R2 Square {model_test_r2:.4f}")

    print('\n')
    
    
    

RandomForestRegressor
model performance for training set
Root Mean Squared error 154184.2997
Mean Squared error 23772798281.7387
Mean Absolute error 39949.4880
R2 Square 0.9710
model performance for testing set
Root Mean Squared error 247752.8537
Mean Squared error 61381476530.9080
Mean Absolute error 105826.3617
R2 Square 0.9184


Linear Regression
model performance for training set
Root Mean Squared error 559313.7144
Mean Squared error 312831831093.0353
Mean Absolute error 268437.6549
R2 Square 0.6183
model performance for testing set
Root Mean Squared error 507556.7988
Mean Squared error 257613903997.1091
Mean Absolute error 281329.9042
R2 Square 0.6575


Ridge
model performance for training set
Root Mean Squared error 559314.5848
Mean Squared error 312832804775.1844
Mean Absolute error 268394.1969
R2 Square 0.6183
model performance for testing set
Root Mean Squared error 507539.6755
Mean Squared error 257596522203.1086
Mean Absolute error 281279.8973
R2 Square 0.6575


Lasso
model 

## Initialize few parameters for Hyperparameter tunning

In [41]:
knn_para = {
    'n_neighbors': [2,3,10,12,15],
}
rf_para ={
    'max_depth':[5,6,8,12,None],
    'max_features':[5,8,"auto",10],
    'min_samples_split':[2,5,7,10],
    'n_estimators': [100,200,500,1000]
}

In [42]:
#Model List for Hyperparameter Tunning
randomcv_models = [
    ('KNN',KNeighborsRegressor(),knn_para),
    ('RF',RandomForestRegressor(),rf_para)
]


In [43]:
from sklearn.model_selection import RandomizedSearchCV

model_param ={}
for name,model,param in randomcv_models:
    random = RandomizedSearchCV(
        estimator=model,
        param_distributions=param,
        n_iter=100,
        cv=3,
        n_jobs=-1)

    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

for i in model_param:
    print(f"------------Best Params----------------")
    print(model_param[i])
    


------------Best Params----------------
{'n_neighbors': 2}
------------Best Params----------------
{'n_estimators': 100, 'min_samples_split': 2, 'max_features': 10, 'max_depth': None}


In [45]:
#Retraining The Models With The Best Parameters
models = {
    'RandomForestRegressor':RandomForestRegressor(n_estimators=100,min_samples_split=2,max_features=10,max_depth=None),
    'KNeighborsRegressor':KNeighborsRegressor(n_neighbors=2),
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)  # Train Model

    #make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #evaluate train test dataset
    model_train_r2, model_train_mae, model_train_mse, model_train_rmse= evaluate_model(y_train,y_train_pred)
    model_test_r2, model_test_mae, model_test_mse,model_test_rmse = evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])

    print('model performance for training set')
    print(f"Root Mean Squared error {model_train_rmse:.4f}")
    print(f"Mean Squared error {model_train_mse:.4f}")
    print(f"Mean Absolute error {model_train_mae:.4f}")
    print(f"R2 Square {model_train_r2:.4f}")

    print('='*35)

    print('model performance for testing set')
    print(f"Root Mean Squared error {model_test_rmse:.4f}")
    print(f"Mean Squared error {model_test_mse:.4f}")
    print(f"Mean Absolute error {model_test_mae:.4f}")
    print(f"R2 Square {model_test_r2:.4f}")

    print('\n')
    
    

RandomForestRegressor
model performance for training set
Root Mean Squared error 123058.9942
Mean Squared error 15143516057.5700
Mean Absolute error 39810.0454
R2 Square 0.9815
model performance for testing set
Root Mean Squared error 237561.8183
Mean Squared error 56435617508.9289
Mean Absolute error 103510.4234
R2 Square 0.9250


KNeighborsRegressor
model performance for training set
Root Mean Squared error 224941.1008
Mean Squared error 50598498818.0217
Mean Absolute error 66339.0655
R2 Square 0.9383
model performance for testing set
Root Mean Squared error 301678.3838
Mean Squared error 91009847240.6196
Mean Absolute error 120410.3860
R2 Square 0.8790


